# Cooperation-Aware Human-Type Inference in a Keys–Doors–Gems Environment

## Tutorial notebook

This notebook introduces the complete pipeline implemented in the accompanying repository. It starts with a grid-world environment, constructs two models of human behaviour, uses observed actions to infer which model better explains the human, and finally uses that belief to plan the assistant's actions.

The notebook is organised into seven sessions. Each session introduces the purpose of the corresponding implementation, its inputs and outputs, the main computational steps, and the mathematical formulation. The repository remains the source of the implementation; this notebook provides the conceptual guide and the experiments that will be added alongside the explanations.

### Pipeline at a glance

1. **KDG Environment Setup** — define the environment and its state-transition rules.
2. **Independent Human Policy** — compute how a human would act when planning alone.
3. **Joint Collaborative Human–Assistant Policy** — compute coordinated behaviour when both agents optimise the team's objective.
4. **Constructing Human–Assistant Policy Mismatch** — combine an independent human with a collaborative assistant and account for resources held by the assistant.
5. **Boltzmann-Rational Trajectory Sampling** — turn the two Q-tables into stochastic behavioural models and sample actions.
6. **Bayesian Human-Type Inference** — compare the likelihood of observed actions under the two models and update the posterior.
7. **Inference-Aware Assistant** — learn an assistant policy conditioned on both the physical state and its current belief about the human.

### Conventions used throughout

- $H$ denotes the human and $A$ denotes the assistant (called the robot in parts of the implementation).
- A human type is fixed within an episode but is hidden from the assistant: independent ($I$) or collaborative ($C$).
- The planning code stores **utility as negative cost-to-go**. Therefore, a larger (less negative) Q-value is better.
- The environment is turn-based: exactly one agent acts at each decision step, and the turn then switches.
- The default task is for the human to collect the requested gem or gems. The assistant can help by moving, collecting keys, and unlocking doors, but cannot collect gems.
- The exact maze, goal specification, random seed, rationality parameter, and planning settings used in an experiment should be stated when presenting its results.

# Session 0 — Repo and Packages Setup

In [ ]:
!git clone https://github.com/g-gaogui/Cooperation-Aware-Human-Type-Inference.git

In [ ]:
!git clone https://github.com/cosilab/CLIPS.jl.git

In [ ]:
import os

# Define your new folder name
folder_name = "Cooperation-Aware-Human-Type-Inference/maze_dataset"

sample_mazes = {
    "maze_1": """
    WWWWWWWWW
    Wm.r...yW
    W..WWWWWW
    W..WWW..W
    W..R....W
    W..WWW.WW
    W..WWW.WW
    W...WW.gW
    W..h.W.WW
    WWWYWW.WW
    W..g....W
    WWWWWWWWW
    """,
    "maze_2": """
    WWWWWWWWWWWW
    W.WgW..W..rW
    W.W.W..WWW.W
    W.WRW......W
    W.....W....W
    W.m.W......W
    W......WYW.W
    W.WW...W.W.W
    Wy.W...WgWhW
    WWWWWWWWWWWW
    """
}
# Create the folder if it doesn't already exist
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Folder '{folder_name}' created successfully!")
else:
    print(f"Folder '{folder_name}' already exists.")

for k, v in sample_mazes.items():
    maze = v.strip()
    with open(os.path.join(folder_name, f"{k}.txt"), "w") as f:
        f.write(maze)


In [ ]:
# Run the full pipeline (Windows).
# Jupyter's `!` waits until the process exits before showing stdout, so
# stream the script output line by line instead.
import os
import subprocess
from pathlib import Path

project_dir = Path("Cooperation-Aware-Human-Type-Inference").resolve()
env = os.environ.copy()
env["NOPAUSE"] = "1"  # do not wait for a keypress at the end of run.bat
env["PYTHONUNBUFFERED"] = "1"

for maze in ["maze_1.txt", "maze_2.txt"]:
    process = subprocess.Popen(
        ["cmd", "/c", "run.bat", "1", maze],
        cwd=project_dir,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"run.bat exited with code {process.returncode}")

In [ ]:
# Run the full pipeline (Linux/macOS).
# Jupyter's `!` waits until the process exits before showing stdout, so
# stream the script output line by line instead.
import os
import subprocess
from pathlib import Path

project_dir = Path("Cooperation-Aware-Human-Type-Inference").resolve()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

for maze in ["maze_1.txt", "maze_2.txt"]:
    process = subprocess.Popen(
        ["bash", "./run.sh", "1", maze],
        cwd=project_dir,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"run.sh exited with code {process.returncode}")

# Session 1 — KDG Environment Setup

## Goal

Establish a common environment and vocabulary for the remaining sessions. The Keys–Doors–Gems (KDG) maze is a discrete grid world in which a human and, when required, an assistant navigate obstacles, collect keys, unlock doors, and ultimately enable the human to collect the target gem or gems.

The environment definition is shared by the single-agent and joint-agent planners. Keeping one transition model is important: differences between the behavioural models should come from their planning objectives and information assumptions, not from inconsistent maze mechanics.

## Maze representation

The maze is loaded from an ASCII text file. Each character represents a cell or an object:

| Symbol | Meaning |
|---|---|
| `W` | Wall |
| `.` or space | Empty traversable cell |
| `h` | Human starting position |
| `m` | Assistant starting position |
| `r`, `y`, `b`, `e`, `p` | Red, yellow, blue, green, and pink keys |
| `R`, `Y`, `B`, `E`, `P` | Corresponding coloured doors |
| `g`, `G` | Gems; gem colours are assigned by their order in the maze (red, yellow, blue, then green) |

Coordinates are represented as $(x,y)$, with $x$ increasing to the right and $y$ increasing downwards. The parser uses one-based coordinates.

## State and action representation

For the independent human environment, a state is represented as

$$
s_H=(p_H,K_H,U,G_H),
$$

where:

- $p_H$: human position;
- $K_H$: keys currently held by the human;
- $U$: doors already unlocked;
- $G_H$: gems collected by the human.

For the joint environment, the state is

$$
s=(p_H,K_H,G_H,p_A,K_A,U,\tau),
$$

where $p_A$ and $K_A$ are the assistant's position and keys, and $\tau\in\{H,A\}$ indicates whose turn it is. Door-unlock status $U$ is shared, while keys and gems are agent-specific.

The available actions include:

- movement: up, down, left, and right;
- pickup: collect a key or (for the human only) a gem;
- unlock: use a matching key to unlock an adjacent door;
- wait: remain in the current state and pass the turn.

A movement or resource action has cost $1$. Waiting has a small positive cost in the single-human and joint models (the implementation uses $0.1$) to discourage unproductive waiting loops. The assistant is not allowed to pick up gems. Agents cannot move into the cell occupied by the other agent.

## Goal condition

A goal specification can request the red gem, the yellow gem, or both. The episode terminates as soon as the human holds every gem in the requested goal set. The assistant has no separate gem-collection goal.

## What this session establishes

The output of this session is the parsed maze object and the transition machinery used by all later sessions. The maze object stores dimensions, walls, start positions, keys, doors, gems, and the mapping between logical object identifiers and planner-facing names. The transition functions define which actions are legal and the resulting state and cost.

Before moving on, verify that the selected maze contains a human start and the required goal gems. Joint-policy and assistant-planning sessions also require an assistant start.

## 5. Execution command

The first step in `run.bat` renders the selected ASCII maze so that its layout can be inspected before solving it:

```bat
REM run.bat — Step 1
julia --project=. .\render_maze.jl "maze_dataset\%MAZE_NAME%" "%FOLDER%\%FOLDER%.png"
```

The maze parser used by the Python solvers is `load_maze()` in `value_iteration.py`. It reads the same ASCII file and converts symbols into walls, keys, doors, gems, and starting positions. The render command is a visual check; it does not itself compute a policy.

## 6. Core implementation: state transitions

The central environment function is `successors()` in `value_iteration.py`. It enumerates the legal actions available to the human and yields each action's next state and cost. The excerpt below shows the movement, key-pickup, and waiting cases; the full function also handles gem pickup and unlocking adjacent doors.

```python
# value_iteration.py — successors(), abridged
(x, y), keys_held, unlocked, gems_held = state

# Move to a passable neighbouring cell.
for name, (dx, dy) in MOVES.items():
    nxt = (x + dx, y + dy)
    if passable(maze, nxt, unlocked):
        yield name, (nxt, keys_held, unlocked, gems_held), 1.0

# Pick up a key located at the current cell, if available.
if key_id is not None:
    yield (
        f"pickup({key_id})",
        (pos, keys_held | {key_id}, unlocked, gems_held),
        1.0,
    )

# Waiting leaves the state unchanged but incurs a small cost.
yield WAIT_ACTION, state, wait_penalty
```

This function defines the transition model used by the single-agent planner. The joint planner has a corresponding `joint_successors()` function, which additionally switches turns and checks collisions.

# Session 2 — Independent Human Policy

## 1. Goal of this session's code

Compute an optimal policy for a human who plans **as an individual**, without modelling the assistant as a strategic collaborator. The human minimises its own cost-to-go to collect the requested gem or gems.

This policy provides the independent-behaviour hypothesis used later for trajectory generation and human-type inference.

## 2. Input and output

**Inputs**

- The parsed KDG maze.
- A goal specification (for example, red, yellow, or both).
- The single-human transition model, including legal actions and their costs.
- The set of key-availability scenarios considered by the solver.

**Outputs**

- A reachable-state utility table $V(s)$.
- A Q-value table $Q(s,a)$ for legal actions.
- A greedy policy selecting the highest-Q action in each state.
- A representative greedy path/trajectory from the initial state, together with summary information such as the path cost and number of value-iteration sweeps.
- A serialised JSON result that later sessions can load.

An important implementation detail is that the solver enumerates all $2^{|K|}$ combinations of initially available key types. The key-availability scenario is included in the serialised state key as `initial_keys`, so otherwise-identical states from different scenarios do not overwrite one another. The displayed baseline path is generated from the all-keys-available scenario.

## 3. How it works in general

The single-human problem is modelled as a deterministic shortest-path Markov decision process over the reachable states.

1. **Construct the state space.** Starting from the human's initial state, explore all states reachable through legal movement, pickup, unlock, and wait actions.
2. **Specify terminal states.** A state is terminal when the human has collected all requested goal gems. Its remaining cost-to-go is zero.
3. **Run value iteration.** Initialise non-terminal values to negative infinity and repeatedly update each state using the best available action. Because the implementation maximises negative cost, actions leading to lower remaining cost receive larger values.
4. **Extract Q-values and a policy.** Once the value function stops changing, calculate the value of every legal action and select the action with the maximum Q-value.
5. **Repeat across key-availability scenarios.** For each subset of key types that may be available, solve the corresponding reachable problem and store the scenario in the state key.
6. **Generate a representative path.** Starting from the initial state in the all-keys-available scenario, repeatedly follow the greedy policy until the goal is reached.

The solver uses synchronous updates: every state in one sweep is updated using the value function from the previous sweep. There is no discount factor in this shortest-path formulation.

## 4. Core mathematical formulation

Let $S$ be the reachable state space, $A(s)$ the legal actions in state $s$, $T(s,a)$ the deterministic successor state, and $c(s,a)$ the immediate action cost.

The terminal condition is

$$
V(s)=0,\qquad s\in S_{\mathrm{goal}}.
$$

For non-terminal states, value iteration uses

$$
V(s)=\max_{a\in A(s)}
\left[-c(s,a)+V(T(s,a))\right].
$$

The corresponding action-value function is

$$
Q(s,a)=-c(s,a)+V(T(s,a)),
$$

and the greedy independent policy is

$$
\pi_I(s)=\arg\max_{a\in A(s)}Q_I(s,a).
$$

Since $Q$ is negative cost-to-go, maximising $Q$ is equivalent to minimising the total remaining cost. The small waiting cost is included in $c(s,a)$ for the wait action.

## 5. Execution command

`run.bat` Step 2 invokes the independent human solver:

```bat
python value_iteration.py --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\value_iteration_results.json"
```

The default goal specification in the script is `both` (red and yellow gems). The solver writes the utility, Q-values, policy, and trajectory information to the JSON file consumed by later sessions.

## 6. Core mathematical implementation

The central routine is `value_iteration()` in `value_iteration.py`. It first enumerates reachable states, initialises goal states to zero utility, and repeatedly applies the Bellman optimality update. The implementation is synchronous: every sweep reads from the previous value table (`V`) and writes to a separate table (`V_new`).

```python
# value_iteration.py — core of value_iteration()
states = explore_reachable(maze, start, unavailable_keys)
V = {s: 0.0 if is_goal(s, goal_gems) else float("-inf")
     for s in states}

for _ in range(max_iters):
    changed = False
    V_new = dict(V)
    for s in states:
        if is_goal(s, goal_gems):
            V_new[s] = 0.0
            continue
        best = max(
            (-cost + V[sp]
             for _, sp, cost in successors(
                 maze, s, unavailable_keys, wait_penalty
             ) if sp in V),
            default=float("-inf"),
        )
        changed |= (best != V[s])
        V_new[s] = best
    V = V_new
    if not changed:
        break

# Recover Q(s,a) and the greedy action after V has converged.
Q[s] = {act: -cost + V[sp]
        for act, sp, cost in successors(maze, s, unavailable_keys, wait_penalty)
        if sp in V}
policy[s] = max(Q[s], key=Q[s].get)
```

The source additionally repeats this solve over all initial key-availability combinations. The `initial_keys` scenario is part of the serialised state key, preventing states from different scenarios from overwriting one another.

# Session 3 — Joint Collaborative Human–Assistant Policy

## 1. Goal of this session's code

Compute an optimal coordinated policy for the human and assistant when both agents cooperate to achieve the human's gem-collection goal while minimising their **combined action cost**.

Unlike the independent model, the collaborative model explicitly reasons about the assistant's position, inventory, and opportunities to help. It can therefore prefer actions that are not optimal for the human considered in isolation—for example, the assistant may collect a key or unlock a door to make the human's route easier.

## 2. Input and output

**Inputs**

- The parsed KDG maze, including both human and assistant starting positions.
- The human's goal gem specification.
- The joint transition model, including turn order, collision constraints, shared unlocked doors, and agent-specific inventories.

**Outputs**

- The reachable joint-state space.
- Joint utility and Q-value tables.
- A greedy joint policy over human-turn and assistant-turn states.
- A coordinated trajectory and path summary from the initial joint state.
- A JSON result consumed by the mismatch, trajectory-sampling, inference, and assistant-planning sessions.

## 3. How it works in general

The joint problem is a turn-based deterministic MDP.

1. **Construct the joint state.** Track the human and assistant positions, their separate key inventories, the human's collected gems, the shared unlocked-door set, and whose turn it is.
2. **Generate legal actions for the acting agent.** Only the agent whose turn it is can move. Movement into the other agent's occupied cell is disallowed. The assistant may collect keys and unlock doors, but may not collect gems.
3. **Switch turns.** After an action, the turn changes to the other agent. Both agents have a wait action, allowing one agent to yield while the other acts.
4. **Define the team objective.** Each real action costs the acting agent one unit. The value-iteration objective minimises the sum of human and assistant action costs. Waiting has a small cost in the implementation.
5. **Explore and solve.** Enumerate reachable joint states and apply value iteration, then calculate Q-values and extract the greedy action at every state.
6. **Generate a coordinated trajectory.** Follow the joint policy from the initial state. The resulting trajectory alternates between human and assistant decisions and ends when the human has collected the requested gems.

The turn-based design avoids the need to define simultaneous-action collision resolution. The unlocked-door set is shared because a door opened by either agent remains open for both; keys and collected gems remain associated with their respective agents.

## 4. Core mathematical formulation

Let $s$ be a joint state, $A(s)$ the legal actions for the agent whose turn it is, and $T(s,a)$ the next joint state. Let $c_H(s,a)$ and $c_A(s,a)$ be the human and assistant costs incurred by the transition.

The terminal condition is

$$
V_C(s)=0,\qquad s\in S_{\mathrm{goal}},
$$

where the goal is determined only by the human's collected gems.

For a non-terminal joint state,

$$
V_C(s)=\max_{a\in A(s)}
\left[-\big(c_H(s,a)+c_A(s,a)\big)+V_C(T(s,a))\right].
$$

The joint Q-value and greedy collaborative policy are

$$
Q_C(s,a)=
-\big(c_H(s,a)+c_A(s,a)\big)+V_C(T(s,a)),
$$

$$
\pi_C(s)=\arg\max_{a\in A(s)}Q_C(s,a).
$$

At a human-turn state, this policy describes the human's action under the assumption of collaboration. At an assistant-turn state, it describes the assistant's coordinated action.

## 5. Execution command

`run.bat` Step 4 runs the joint solver. (Step 3 between Sessions 2 and 3 visualises the independent trajectory.)

```bat
python joint_value_iteration.py --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\joint_value_iteration_results.json"
```

The command uses the same maze file, but the joint solver starts from the state containing both agents and writes a joint-state Q-table.

## 6. Core mathematical implementation

The joint transition model is implemented by `joint_successors()` in `joint_value_iteration.py`. It generates actions for the agent whose turn it is, assigns the cost to the acting agent, and changes the turn. The value-iteration routine then minimises the **combined** human and assistant cost by maximising its negative.

```python
# joint_value_iteration.py — Bellman update in joint_value_iteration()
for s in states:
    if s in goal_states:
        V_new[s] = 0.0
        continue

    best = float("-inf")
    for _, sp, human_cost, assistant_cost in successors[s]:
        candidate = -(human_cost + assistant_cost) + V[sp]
        best = max(best, candidate)
    V_new[s] = best

V = V_new

# Recover the joint action values.
Q[s] = {
    action: -(human_cost + assistant_cost) + V[sp]
    for action, sp, human_cost, assistant_cost in successors[s]
}
policy[s] = max(Q[s], key=Q[s].get)
```

The distinction from Session 2 is the objective and state representation: the planner now sees both agents, while the one-step reward is the negative sum of their action costs.

# Session 4 — Constructing Human–Assistant Policy Mismatch

## 1. Goal of this session's code

Construct and simulate a **mixed-policy interaction** in which the human follows the independent policy while the assistant follows the collaborative policy.

This deliberately combines two different behavioural assumptions:

- The human chooses actions using the single-human Q-table and does not re-plan to account for the assistant's intentions.
- The assistant chooses actions using the joint collaborative Q-table and therefore plans to help the human.

The purpose is to make the mismatch explicit and to produce a realistic joint trajectory for a non-collaborative human interacting with a cooperative assistant.

## 2. Input and output

**Inputs**

- The single-human value-iteration JSON from Session 2.
- The joint human–assistant value-iteration JSON from Session 3.
- The maze and human goal specification.
- A maximum rollout length.

**Outputs**

- A mixed-policy trajectory containing the joint state, acting agent, selected action, Q-values and relevant state keys.
- The sequence of human and assistant actions.
- Human, assistant, and combined realised action costs when the goal is reached.
- The source Q-table used to select each action, allowing the two policy sources to be distinguished.

## 3. How it works in general

At each turn, the rollout selects an action from the Q-table associated with the acting agent.

**Human turn**

1. Project the current joint state into the single-human state representation.
2. Retrieve the corresponding independent-human Q-values.
3. Rank candidate actions by Q-value, from highest to lowest.
4. Check the candidates against the actual joint transition model. If an action is unavailable or would cause a collision, skip it and try the next-ranked action.
5. Execute the first legal candidate.

The human's Q-ranking remains independent: the collision check is a physical feasibility constraint, not a re-optimisation of the human's objective.

**Assistant turn**

1. Retrieve the collaborative Q-values using the full joint state.
2. Rank assistant actions by Q-value.
3. Select the highest-ranked action that is executable in the current joint state.
4. Execute it and switch back to the human turn.

### Why an extended human Q-value state is necessary

The single-human Q-table is not indexed only by the human's current position and inventory. Its state key also records `initial_keys`, which identifies the key types assumed to be available in the world for that value-iteration run. Session 2 precomputes Q-values for every key-availability combination.

This matters when the assistant holds a key. The human's independent model should not behave as if that key were still available for the human to pick up. The mixed rollout therefore projects the joint state into a human-visible key-availability scenario:

- Start from all key types in the maze.
- Exclude key types held by the assistant.
- Add back the key types corresponding to doors that have already been unlocked, because the key's function has already been realised in the shared environment.

The projected single-human state key is consequently extended with the relevant `initial_keys` set. This lets the rollout retrieve the appropriate precomputed human Q-values without pretending that the assistant-held key is available to the human. The human's current held keys, collected gems, position, and shared unlocked doors are also retained.

## 4. Core mathematical formulation

Let $s$ be the current joint state. Define a projection into the independent-human Q-table state:

$$
\phi_H(s)=
\left(
p_H,K_H,U,G_H,K_{\mathrm{visible}}^{\,0}(s)
\right),
$$

where the human-visible initial key set is

$$
K_{\mathrm{visible}}^{\,0}(s)
=
\big(K_{\mathrm{all}}\setminus K_A^{\mathrm{held}}\big)
\cup K_{\mathrm{unlocked}}.
$$

Here $K_{\mathrm{unlocked}}$ denotes the key types whose corresponding doors have already been unlocked. The extended state key distinguishes different initial key-availability scenarios.

At a human turn, the mixed rollout selects the highest-valued executable action under the independent model:

$$
a_H^*=
\arg\max_{a\in A_H^{\mathrm{legal}}(s)}
Q_I\big(\phi_H(s),a\big).
$$

At an assistant turn, it selects the highest-valued executable action under the collaborative model:

$$
a_A^*=
\arg\max_{a\in A_A^{\mathrm{legal}}(s)}
Q_C(s,a).
$$

These maximisations are subject to the actual joint environment's legality and collision constraints. The resulting policy is mixed: it is not the greedy policy of one unified objective.

## 5. Execution command

`run.bat` Step 6 runs the mixed-policy rollout after the independent and joint Q-tables have been generated:

```bat
python mismatch.py --single "%FOLDER%\value_iteration_results.json" --joint "%FOLDER%\joint_value_iteration_results.json" --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\mismatch_results.json"
```

The command loads both policies and simulates a trajectory. Step 5 in `run.bat` visualises the joint collaborative trajectory; Step 7 visualises the mixed-policy result.

## 6. Core implementation: selecting actions from different models

The key routines are `choose_human_action()` and `choose_assistant_action()` in `mismatch.py`. The human's state is projected into the single-agent representation, then actions are ranked using the independent Q-table. The selected action must still be executable in the actual joint state and must not move the human onto the assistant's cell. The assistant, in contrast, ranks actions using the joint collaborative Q-table.

```python
# mismatch.py — essential human-action selection logic
initial_keys = human_visible_initial_keys(maze, keysA, unlocked)
h_key = single_state_key(posH, keysH, unlocked, gemsH, initial_keys)
q_values = get_q_values(single_q, h_key)
ranked = sorted(q_values.items(), key=lambda item: item[1], reverse=True)

# Only execute an action that is legal in the joint environment.
successors = {
    action: (next_state, hc, ac)
    for action, next_state, hc, ac in joint_successors(maze, state)
    if action.startswith("H:")
}
for raw_action, _q in ranked:
    joint_action = f"H:{raw_action}"
    if joint_action not in successors:
        continue
    next_state, hc, ac = successors[joint_action]
    if next_state[0] == posA:       # collision constraint
        continue
    return joint_action, next_state, hc, ac, q_values, "single_human_q"
```

```python
# mismatch.py — essential assistant-action selection logic
q_values = get_q_values(joint_q, joint_state_key(state))
ranked = sorted(q_values.items(), key=lambda item: item[1], reverse=True)
successors = {
    action: (next_state, hc, ac)
    for action, next_state, hc, ac in joint_successors(maze, state)
    if action.startswith("A:")
}
for action, _q in ranked:
    if action in successors:
        next_state, hc, ac = successors[action]
        return action, next_state, hc, ac, q_values, "joint_q"
```

The implementation also constructs the human's `initial_keys` view from keys held by the assistant and doors already unlocked. This is essential: a key taken by the assistant is no longer available for the human to pick up, while an already-unlocked door no longer requires that key.

# Session 5 — Boltzmann-Rational Trajectory Sampling

## 1. Goal of this session's code

Convert the deterministic Q-value tables into probabilistic models of human action selection, then sample stochastic trajectories under each behavioural hypothesis.

A deterministic greedy policy always selects a maximum-Q action. A Boltzmann-rational model instead assigns higher probability to better actions while allowing lower-valued actions to occur. The parameter $\beta$ controls how strongly the human favours high-Q actions.

This gives the inference session sequences of actions that are plausible under either independent or collaborative behaviour.

## 2. Input and output

**Inputs**

- The maze and human goal specification.
- The single-human Q-table from Session 2 for independent sampling.
- The joint Q-table from Session 3 for collaborative sampling.
- A non-negative rationality parameter $\beta$.
- A maximum number of human actions and an optional random seed.

**Outputs**

- An independently generated human action sequence.
- A collaboratively generated human action sequence.
- JSON trajectory files containing the sampled action paths and relevant sampling metadata.

In the collaborative sampler, the joint Q-values determine the human's action distribution at human-turn states. For trajectory generation, the assistant is explicitly assumed to wait after each human action. The sampler therefore isolates the human's behaviour under the collaborative model; it does **not** execute the assistant's greedy collaborative policy.

## 3. How it works in general

The two samplers follow the same broad procedure but query different Q-tables.

1. Initialise the environment and load the appropriate Q-table.
2. At the current human decision state, retrieve the action Q-values.
3. Remove unavailable or non-finite actions.
4. Convert the remaining Q-values into a probability distribution using a Boltzmann softmax.
5. Randomly sample one action according to that distribution.
6. Apply the action using the environment's transition function.
7. For collaborative sampling, apply an assistant wait transition before the next human decision, unless the goal has already been reached.
8. Stop when the human reaches the goal or the maximum trajectory length is reached.

The independent sampler advances the single-human state directly. The collaborative sampler maintains the joint state so that it can look up the human's Q-values in the joint collaborative table, while keeping the assistant stationary through explicit wait actions.

A fixed random seed makes a sampled trajectory reproducible. Changing the seed may produce a different trajectory even when the Q-table and $\beta$ remain unchanged.

## 4. Core mathematical formulation

For a behavioural hypothesis $Z\in\{I,C\}$, let $Q_Z(s,a)$ denote the relevant human action-value function. The Boltzmann-rational action model is

$$
\pi_Z(a\mid s;\beta)
=
\frac{\exp\left(\beta Q_Z(s,a)\right)}
{\displaystyle\sum_{a'\in A_Z(s)}
\exp\left(\beta Q_Z(s,a')\right)}.
$$

The distribution is normalised over the legal actions included by the corresponding sampler.

The rationality parameter has the following interpretation:

- $\beta=0$: all included legal actions are equally likely.
- Larger $\beta$: higher-Q actions receive more probability.
- In the limit as $\beta$ becomes large, probability concentrates on maximum-Q actions (with ties sharing the probability).

Because the implementation uses negative cost-to-go Q-values, a higher Q-value means a lower expected remaining cost. For numerical stability, the implementation subtracts the maximum logit before exponentiation; this does not change the resulting probabilities.

## 5. Execution command

`run.bat` Step 8 generates two observed human trajectories, both with $\beta=20$. One samples from the independent Q-table; the other samples human actions from the joint collaborative Q-table.

```bat
python sample_single_trajectory.py --q_file "%FOLDER%\value_iteration_results.json" --maze_file "maze_dataset\%MAZE_NAME%" --out_file "%FOLDER%\observed_ind_H_trajectory.json" --beta 20
python sample_joint_trajectory.py --q_file "%FOLDER%\joint_value_iteration_results.json" --maze_file "maze_dataset\%MAZE_NAME%" --out_file "%FOLDER%\observed_col_H_trajectory.json" --beta 20
```

The collaborative sampler assumes that the assistant waits after each human action. It samples the human's action from the joint Q-values; it does not execute the assistant's greedy joint policy during this sampling procedure.

## 6. Core mathematical implementation: Boltzmann action sampling

Both samplers convert Q-values into a categorical distribution and draw one action. The implementation uses a numerically stable softmax by subtracting the maximum scaled Q-value before exponentiation.

```python
# sample_single_trajectory.py — boltzmann_probabilities()
valid = {a: float(q) for a, q in q_dict.items()
         if q is not None and math.isfinite(float(q))}

if beta == 0:
    p = 1.0 / len(valid)
    return {a: p for a in valid}

scaled = {a: beta * q for a, q in valid.items()}
max_scaled = max(scaled.values())
weights = {a: math.exp(v - max_scaled) for a, v in scaled.items()}
total = sum(weights.values())
probabilities = {a: w / total for a, w in weights.items()}

action = random.choices(
    list(probabilities), weights=list(probabilities.values()), k=1
)[0]
```

Because the Q-values are negative cost-to-go values, a larger Q-value represents a lower expected remaining cost and receives a larger probability. At $\beta=0$ the distribution is uniform; increasing $\beta$ makes it more concentrated on high-Q actions.

# Session 6 — Bayesian Human-Type Inference

## 1. Goal of this session's code

Infer whether an observed human action sequence is more consistent with independent or collaborative behaviour.

The assistant does not directly observe the human's type. Instead, it maintains two competing behavioural hypotheses and evaluates how likely the observed actions would be under each hypothesis. Evidence is accumulated across the trajectory and combined with a prior belief using Bayes' rule.

This session performs **trajectory evaluation**: it scores an observed sequence rather than selecting actions for the human.

## 2. Input and output

**Inputs**

- The maze and human goal specification.
- The independent-human Q-table from Session 2.
- The collaborative joint Q-table from Session 3.
- An observed sequence of human actions, such as a trajectory generated in Session 5.
- The Boltzmann rationality parameter $\beta$.
- Prior probabilities for the independent and collaborative hypotheses.

**Outputs**

- The probability of each observed action under the independent model.
- The probability of each observed action under the collaborative model.
- Cumulative log-likelihoods under both hypotheses.
- The cumulative log-likelihood ratio.
- The posterior probability of collaborative behaviour (and, by complement, independent behaviour) after each observation.
- A trajectory record that tracks the state and inference quantities over time.

The implementation assumes the assistant waits between successive observed human actions. This keeps each observation aligned with a human-turn state and matches the trajectory-generation setup in Session 5.

## 3. How it works in general

1. **Initialise the hypotheses.** Set prior probabilities $P(I)$ and $P(C)$, which are normalised to sum to one.
2. **Reconstruct the current state.** Start from the initial joint state and process the observed human actions in order.
3. **Evaluate the action under the independent model.** Project the joint state into the independent-human state key, retrieve the independent Q-values, retain executable human actions, and calculate their Boltzmann probabilities.
4. **Evaluate the action under the collaborative model.** Retrieve the joint Q-values for the current joint state, retain executable human actions, and calculate the corresponding Boltzmann probabilities.
5. **Accumulate evidence.** Add the log probability of the observed action to each hypothesis's cumulative log-likelihood.
6. **Calculate the log-likelihood ratio and posterior.** Positive log-likelihood ratio indicates evidence for collaboration; negative values indicate evidence for independence. Combine the evidence with the prior to obtain the posterior.
7. **Advance the environment.** Apply the observed human action. If another human action remains, apply an assistant wait transition to reach the next human-turn state.

The observed action is executed in the environment after it has been evaluated. The inference routine does not replace the observed action with the action preferred by either Q-table.

## 4. Core mathematical formulation

Let the observed human trajectory be $\mathcal{D}_{1:T}=(a_1,\ldots,a_T)$, and let $s_t$ be the state at which action $a_t$ is observed. Under hypothesis $Z\in\{I,C\}$, the trajectory likelihood is

$$
P(\mathcal{D}_{1:T}\mid Z)
=
\prod_{t=1}^{T}
\pi_Z(a_t\mid s_t;\beta).
$$

The cumulative log-likelihood is

$$
\ell_Z(T)
=
\sum_{t=1}^{T}
\log \pi_Z(a_t\mid s_t;\beta).
$$

The cumulative log-likelihood ratio, with the collaborative model in the numerator, is

$$
\operatorname{LLR}_T
=
\ell_C(T)-\ell_I(T).
$$

Thus, a positive LLR favours the collaborative hypothesis, while a negative LLR favours the independent hypothesis.

Bayes' rule gives

$$
P(C\mid\mathcal{D}_{1:T})
=
\frac{
P(C)\exp(\ell_C(T))
}{
P(I)\exp(\ell_I(T))
+
P(C)\exp(\ell_C(T))
}.
$$

Equivalently, in terms of the LLR,

$$
P(C\mid\mathcal{D}_{1:T})
=
\frac{1}{
1+\exp\left(
-\left[
\log\frac{P(C)}{P(I)}
+\operatorname{LLR}_T
\right]
\right)
}.
$$

The log-space calculation avoids directly multiplying many small action probabilities, which can cause numerical underflow for long trajectories. An action assigned zero probability by a hypothesis contributes a log-likelihood of negative infinity under that hypothesis.

## 5. Execution command

`run.bat` Step 9 evaluates both sampled trajectories. The first call evaluates the independently sampled trajectory; the second evaluates the collaboratively sampled trajectory.

```bat
python predict.py --single "%FOLDER%\value_iteration_results.json" --joint "%FOLDER%\joint_value_iteration_results.json" --trajectory "%FOLDER%\observed_ind_H_trajectory.json" --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\predicted_type_single.json" --beta 20
python predict.py --single "%FOLDER%\value_iteration_results.json" --joint "%FOLDER%\joint_value_iteration_results.json" --trajectory "%FOLDER%\observed_col_H_trajectory.json" --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\predicted_type_joint.json" --beta 20
```

Step 10 then visualises the posterior predictions. The inference routine scores the observed action sequence; it does not choose replacement actions for the human.

## 6. Core mathematical implementation: likelihood accumulation and Bayes' rule

In `predict.py`, `infer_human_trajectory()` obtains the probability of each observed action under both models, adds the log probabilities to cumulative log-likelihoods, and calls `posterior_from_log_likelihoods()` to obtain the posterior probability of collaborative behaviour.

```python
# predict.py — core update for one observed action
log_likelihood_I += safe_log_probability(p_independent)
log_likelihood_C += safe_log_probability(p_collaborative)

log_lr = log_likelihood_C - log_likelihood_I
posterior_collaborative = posterior_from_log_likelihoods(
    log_likelihood_I,
    log_likelihood_C,
    prior_independent,
    prior_collaborative,
)
```

The posterior helper performs the calculation in log space:

```python
# predict.py — posterior_from_log_likelihoods(), core calculation
log_I = math.log(prior_independent) + log_likelihood_independent
log_C = math.log(prior_collaborative) + log_likelihood_collaborative

difference = log_I - log_C
if difference >= 0:
    e = math.exp(-difference)
    posterior_C = e / (1.0 + e)
else:
    e = math.exp(difference)
    posterior_C = 1.0 / (1.0 + e)
```

A positive log-likelihood ratio favours the collaborative model. The log-space calculation avoids multiplying many small action probabilities directly and is more numerically stable.

# Session 7 — Inference-Aware Assistant

## 1. Goal of this session's code

Close the observation–inference–assistance loop by learning an assistant policy that depends on both the physical joint state and the assistant's current belief about the human's type.

The assistant must decide how to help while accounting for uncertainty about whether the human behaves independently or collaboratively. As the human acts, the assistant updates its belief; that updated belief can change which assistant action is most useful.

The implementation uses Real-Time Dynamic Programming (RTDP) to learn a belief-conditioned assistant policy.

## 2. Input and output

**Inputs**

- The maze and human goal specification.
- The independent-human Q-table from Session 2.
- The collaborative joint Q-table from Session 3.
- An additional assistant Q-table computed with `fixed_human_value_iteration.py`. This supporting solver fixes the human to the deterministic greedy independent policy recovered from the independent-human Q-table, then optimises the assistant's actions against that fixed human behaviour.
- An initial probability $P(I)$ (with $P(C)=1-P(I)$).
- The Boltzmann rationality parameter $\beta$, which determines the human action models used during training and testing.
- RTDP settings, including the number of trials, maximum steps per trial, random seed, belief-grid resolution, and convergence-check settings.

**Outputs**

- A learned utility table and assistant Q-values indexed by joint state and discretised belief.
- A greedy assistant policy indexed by the same pair.
- Training statistics, including episode counts, number of backups, visited belief states, and convergence information.
- Test rollouts against a simulated independent human and a simulated collaborative human, including actions, costs, trajectories, and the final assistant belief.

The training output is a policy, not a single trajectory. The test phase evaluates that policy separately against each fixed human type.

## 3. How it works in general

### A. Prepare assistant value estimates

The assistant needs an initial estimate of how useful its actions are under the two behavioural hypotheses.

- The independent-human assistant Q-table is produced by `fixed_human_value_iteration.py`. It recovers the human's deterministic independent policy from the single-human Q-table, holds that policy fixed, and optimises the assistant's actions in the resulting joint environment.
- The collaborative assistant Q-table is taken from the assistant-turn entries of the joint collaborative Q-table from Session 3.

These tables provide heuristic initial values for RTDP. They are not the final belief-aware policy.

### B. Represent uncertainty as part of the planning state

The planning state is augmented with the current belief $b=P(I)$. The assistant therefore plans over pairs $(s,b)$, not just physical states $s$.

The belief is projected to the nearest point on a uniform grid. With $N$ bins, the grid has $N+1$ points from zero to one. This makes the belief-augmented state space finite and allows the learned policy to be stored and queried.

### C. Model the human's action and update the belief

At a human turn, calculate the independent and collaborative Boltzmann action distributions from their respective Q-tables. Mix these distributions according to the current belief to obtain the predictive action distribution.

The human's true type is fixed within each training or test episode, but hidden from the assistant. During training, episodes are sampled in a balanced manner across the two true types to provide trajectory coverage. The assistant nevertheless updates its belief using both hypotheses.

### D. Perform RTDP backups

At an assistant turn, enumerate legal assistant actions and select the action with the largest current Q-value. Update the value of the visited belief-augmented state using a Bellman backup.

At a human turn, the next human action is stochastic under the belief-weighted mixture model. The value update therefore takes an expectation over possible human actions. Each possible action leads to a successor physical state and a successor belief obtained by Bayes' rule.

The algorithm performs these updates along sampled trajectories rather than sweeping over the entire belief-augmented state space. Unvisited values are initialised using a belief-weighted heuristic based on the independent and collaborative assistant Q-tables.

### E. Test the learned policy

After training, look up the assistant action using the current joint state and discretised belief. At human turns, simulate the selected true human type, update the assistant's belief from the observed action, and continue. The test routine runs separate episodes for independent and collaborative humans so that the same learned assistant policy can be evaluated under both conditions.

If an exact state-and-belief policy entry is unavailable during testing, the lookup first tries the nearest stored belief for the same physical state. This fallback is recorded in the test result.

## 4. Core mathematical formulation

Let $b=P(I\mid\mathcal{D})$ be the assistant's current belief that the human is independent. The complementary collaborative belief is $1-b$.

### Predictive human action distribution

The assistant's predictive distribution over the next human action is the mixture

$$
P(a\mid s,b)
=
b\,\pi_I(a\mid s;\beta)
+
(1-b)\,\pi_C(a\mid s;\beta).
$$

### Bayesian belief transition

After observing human action $a$, the belief is updated as

$$
b' =
P(I\mid a,s,b)
=
\frac{
b\,\pi_I(a\mid s;\beta)
}{
b\,\pi_I(a\mid s;\beta)
+
(1-b)\,\pi_C(a\mid s;\beta)
}.
$$

The physical state also transitions to $s'=T(s,a)$. In the implementation, $b'$ is then projected onto the nearest belief-grid point.

### Assistant Bellman backup

At an assistant-turn state, the assistant chooses an action to maximise expected negative total cost:

$$
Q_A(s,b,a)
=
-\big(c_H(s,a)+c_A(s,a)\big)
+
V(s',b),
$$

$$
V(s,b)=\max_{a\in A_A(s)}Q_A(s,b,a).
$$

At a human-turn state, the assistant does not choose the human's action. It evaluates the expectation over the belief-weighted human action distribution:

$$
V(s,b)
=
\sum_{a\in A_H(s)}
P(a\mid s,b)
\left[
-\big(c_H(s,a)+c_A(s,a)\big)
+
V\big(T(s,a),b'(s,a,b)\big)
\right].
$$

These equations describe the belief-dependent Bellman model used by the RTDP backups. The implementation stores values and Q-values only for visited state–belief pairs and initialises unseen assistant-turn values with a heuristic.

### Heuristic initialisation

For an assistant action $a$, let $Q_I^A(s,a)$ be the assistant Q-value against the fixed independent human policy, and $Q_C^A(s,a)$ the collaborative assistant Q-value. The initial belief-weighted action estimate is

$$
\widehat Q_A(s,b,a)
=
bQ_I^A(s,a)+(1-b)Q_C^A(s,a).
$$

The initial value estimate uses a **shared action**:

$$
\widehat V(s,b)
=
\max_{a\in A_A(s)}
\left[
bQ_I^A(s,a)+(1-b)Q_C^A(s,a)
\right].
$$

This is important: the assistant must choose one action under uncertainty. It should therefore maximise the belief-weighted value of each common action, rather than independently maximising each hypothesis and combining the two maxima.

## Closing the loop

The complete pipeline is now:

1. The environment provides the current physical state.
2. The human acts according to one fixed but hidden behavioural type.
3. The assistant evaluates the observed action under both behavioural models.
4. Bayes' rule updates the assistant's belief.
5. The assistant selects an action using its learned policy for the current physical state and belief.
6. The environment transitions and the process repeats until the human reaches the goal or the rollout terminates.

The central distinction from Session 6 is that Bayesian inference is no longer only an offline evaluation of a completed trajectory. In this session, belief is part of the assistant's planning state and can influence its next action.

## 5. Execution commands

Session 7 has a supporting optimisation step followed by RTDP training, testing, and visualisation. `run.bat` Steps 11–13 are:

```bat
REM Step 11: optimise the assistant while fixing the human to the independent policy
python fixed_human_value_iteration.py --maze "maze_dataset\%MAZE_NAME%" --out "%FOLDER%\col_A_against_ind_H.json" --human-q-values "%FOLDER%\value_iteration_results.json"

REM Step 12: train the belief-aware assistant, then test against both human types
python belief_aware_assistant_rtdp.py train --maze "maze_dataset\%MAZE_NAME%" --independent-human-q-values "%FOLDER%\value_iteration_results.json" --joint-q-values "%FOLDER%\joint_value_iteration_results.json" --assistant-independent-q-values "%FOLDER%\col_A_against_ind_H.json" --out "%FOLDER%\belief_aware_assistant_rtdp.json" --seed 42 --beta 20
python belief_aware_assistant_rtdp.py test --maze "maze_dataset\%MAZE_NAME%" --out-dir "%FOLDER%" --independent-human-q-values "%FOLDER%\value_iteration_results.json" --joint-q-values "%FOLDER%\joint_value_iteration_results.json" --policy "%FOLDER%\belief_aware_assistant_rtdp.json" --seed 42

REM Step 13: visualise the two test rollouts
julia --project=. rtdp_visualize_trajectory.jl "maze_dataset\%MAZE_NAME%" "%FOLDER%\belief_aware_assistant_rtdp_test_collaborative.json" "%FOLDER%" "belief_aware_assistant_rtdp_test_collaborative.gif"
julia --project=. rtdp_visualize_trajectory.jl "maze_dataset\%MAZE_NAME%" "%FOLDER%\belief_aware_assistant_rtdp_test_independent.json" "%FOLDER%" "belief_aware_assistant_rtdp_test_independent.gif"
```

Step 11 is a required supporting solve: it produces an assistant Q-table against a fixed deterministic independent-human policy. Step 12 trains the belief-conditioned policy and then evaluates it separately against each fixed human type.

## 6. Core mathematical implementation: belief update and RTDP backups

The implementation is in `belief_aware_assistant_rtdp.py`. The belief is stored as $p_i=P(I)$. After observing action $a$, Bayes' rule updates it using the two human action models:

```python
# belief_aware_assistant_rtdp.py — update_belief()
def update_belief(p_i, p_action_i, p_action_c):
    denom = p_i * p_action_i + (1.0 - p_i) * p_action_c
    if denom <= 0.0:
        return p_i
    return min(1.0, max(0.0, p_i * p_action_i / denom))
```

At an assistant turn, RTDP backs up the best available assistant action. At a human turn, it forms the belief-weighted mixture of the two human action distributions, then takes the expectation over possible human actions and their corresponding posterior beliefs.

```python
# belief_aware_assistant_rtdp.py — core RTDP backup logic
if state[-1] == "assistant":
    action_q = {
        action: -(ch + ca) + value(sp, p_i)
        for action, sp, ch, ca in assistant_actions(maze, state)
    }
    best_action = max(action_q, key=action_q.get)
    values[k] = action_q[best_action]
else:
    pi, pc = human_action_distributions(
        maze, state, q_h_ind, q_h_joint, beta
    )
    mixture = normalize({
        a: p_i * pi.get(a, 0.0) + (1.0 - p_i) * pc.get(a, 0.0)
        for a in (set(pi) | set(pc))
    })

    expected_value = 0.0
    for action, probability in mixture.items():
        sp, ch, ca = legal_human_actions(maze, state)[action]
        p_next = project_belief(update_belief(
            p_i, pi.get(action, 0.0), pc.get(action, 0.0)
        ))
        expected_value += probability * (-(ch + ca) + value(sp, p_next))
    values[k] = expected_value
```

The value function is indexed by both the physical joint state and a discretised belief. The RTDP implementation initialises unseen assistant-turn states with a belief-weighted heuristic derived from the fixed-independent-human assistant Q-table and the collaborative Q-table. For each candidate action it mixes the two Q-values first, then maximises over a **single shared action**; it does not separately maximise each table before taking the weighted average.